# 04 — Monthly CMAM Temperature Wavelet Analysis

This notebook performs a continuous wavelet transform on the processed monthly temperature perturbations. It uses the actual monthly sampling interval in days; it must not be treated as daily data.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
CSV = PROJECT_ROOT / "data" / "processed" / "temperature_perturbation.csv"
sys.path.insert(0, str(PROJECT_ROOT / "wave_python"))
from waveletFunctions import wavelet, wave_signif

if not CSV.exists():
    raise FileNotFoundError(f"{CSV} not found. Run the temperature preparation script first.")
df = pd.read_csv(CSV)
df["date"] = pd.to_datetime(df["date"])
value_col = next(c for c in ["temperature_perturbation_K", "temperature_perturbation"] if c in df.columns)
df[value_col] = pd.to_numeric(df[value_col], errors="raise")
df = df.sort_values("date")


In [ ]:
dates = pd.DatetimeIndex(df["date"])
temperature = df[value_col].to_numpy(dtype=float)
if len(temperature) < 24:
    raise ValueError("Too few monthly observations for this analysis.")
if not np.isfinite(temperature).all():
    raise ValueError("Non-finite temperature values found.")
dt = 365.2425 / 12
pad, dj, s0 = 1, 0.25, 2 * dt
mother, param = "MORLET", 6
j1 = int(np.floor(np.log2(len(temperature) * dt / s0) / dj))
wave, period, scale, coi = wavelet(temperature, dt, pad, dj, s0, j1, mother, param)
power = np.abs(wave) ** 2
lag1 = np.corrcoef(temperature[:-1], temperature[1:])[0, 1]
signif = np.asarray(wave_signif(temperature, dt=dt, scale=scale, sigtest=0, lag1=lag1, siglvl=0.95, dof=-1, mother=mother, param=param)).squeeze()
print("Samples:", len(temperature), "Lag-1:", lag1)
print("Period range (days):", period.min(), period.max())


In [ ]:
period_mask = (period >= 60) & (period <= 730)
p = period[period_mask]
positive = power[period_mask][power[period_mask] > 0]
vmin, vmax = positive.min(), positive.max()
from matplotlib.colors import LogNorm
fig, ax = plt.subplots(figsize=(12, 6))
mesh = ax.contourf(dates, p, power[period_mask], levels=np.logspace(np.log10(vmin), np.log10(vmax), 40), norm=LogNorm(vmin=vmin, vmax=vmax), cmap="turbo", extend="both")
ratio = power / np.asarray(signif).reshape(-1, 1)
ax.contour(dates, p, ratio[period_mask], levels=[1], colors="black", linewidths=1)
ax.plot(dates, coi, "w--", label="COI")
ax.set_yscale("log")
ax.set_ylim(60, 730)  # short periods bottom, long periods top
ax.set_xlabel("Date")
ax.set_ylabel("Period (days)")
ax.set_title("CMAM temperature-perturbation wavelet power")
fig.colorbar(mesh, ax=ax, label="Wavelet power (log scale)")
ax.legend()
fig.tight_layout()
plt.show()


## Interpretation cautions

With 66 monthly points, the series is short for robustly resolving periods near or above one year. Check whether a feature lies inside the reliable side of the COI, and avoid interpreting a global peak as proof of a persistent physical cycle.
